# Orfipy ORF Prediction

![Orfipy ORF Prediction](https://proto-bio.github.io/proto-assets/images/tool/orfipy/hero.png)

This notebook demonstrates `run_orfipy_prediction`, which identifies open reading frames in DNA sequences by scanning for start and stop codons across all reading frames. It covers single-sequence prediction, custom codon and strand configuration, batch processing, and result inspection.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("orfipy")
display_overview("orfipy")
display_docs_section("orfipy", "Background")

# ORFipy

ORFipy is a fast, flexible ORF (Open Reading Frame) prediction tool that identifies potential coding regions in DNA sequences based on start and stop codons. Unlike gene prediction tools like Prodigal, ORFipy performs simple ORF finding without machine learning - it reports all ORFs matching your criteria, making it ideal for exploratory analysis and custom filtering.

## Background

**What is an ORF?**
An [Open Reading Frame](https://en.wikipedia.org/wiki/Open_reading_frame) is a stretch of DNA between a [start codon](https://en.wikipedia.org/wiki/Start_codon) and an in-frame [stop codon](https://en.wikipedia.org/wiki/Stop_codon):
- **Start codons**: ATG (standard), GTG, TTG (alternative)
- **Stop codons**: TAA (ochre), TAG (amber), TGA (opal)
- **[Reading frames](https://en.wikipedia.org/wiki/Reading_frame)**: 3 forward (+1, +2, +3) and 3 reverse (-1, -2, -3)

**ORF vs Gene:**
- **ORF**: Any sequence matching start/stop pattern (may include non-coding)
- **Gene**: Biologically functional coding sequence (subset of ORFs)

ORFipy finds ORFs; gene predictors like Prodigal use additional signals to distinguish real genes.

## Available tools

In [2]:
display_available_tools("orfipy")

- **`run_orfipy_prediction()`** — ORF (Open Reading Frame) prediction using Orfipy

### `run_orfipy_prediction`

`run_orfipy_prediction` scans DNA sequences for open reading frames using Orfipy, a fast Cython-based ORF finder. For each input sequence it returns all ORFs that pass the configured start codon, stop codon, length, and strand filters. Each predicted ORF carries the amino acid translation, nucleotide sequence, 1-indexed coordinates, strand, reading frame, and GC content. Batch inputs are processed together and results are returned as a nested list keyed by input sequence, so downstream code can group or filter ORFs by their parent sequence.

In [3]:
from proto_tools import OrfipyInput, OrfipyConfig, OrfipyOutput, run_orfipy_prediction

In [4]:
# Display input docs
display_api_reference("orfipy", "input", "run_orfipy_prediction")

# A short DNA sequence with a known ORF (hemoglobin alpha fragment)
inputs = OrfipyInput(sequences="ATGGTGCTGAGCCCGGCGGACAAGACCAACGTGAAGGCGGCGTGGGGCAAGTGA")

**Input** — `OrfipyInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | DNA sequence(s) to analyze for open reading frames |

In [5]:
# Display config docs
display_api_reference("orfipy", "config", "run_orfipy_prediction")

# Forward strand only, ATG start codon only, minimum 30 nucleotides | see docs above for all fields
config = OrfipyConfig(
    start_codons=["ATG"],
    min_len=30,
    strand="f",
    include_stop=True,
)

**Config** — `OrfipyConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `threads` | `int` | `4` | CPU threads passed to orfipy --procs |
| `start_codons` | `list[Literal['ATG', 'GTG', 'TTG', 'CTG']]` | `['ATG', 'GTG', 'TTG']` | Start codons to recognize for ORF prediction |
| `stop_codons` | `list[Literal['TAA', 'TAG', 'TGA']]` | `['TAA', 'TAG', 'TGA']` | Stop codons to recognize for ORF prediction |
| `strand` | `Literal['f', 'r', 'b']` | `'b'` | Which strand(s) to scan: 'f' (forward), 'r' (reverse), or 'b' (both) |
| `min_len` | `int` | `0` | Min ORF length in nt; 0 keeps all ORFs |
| `max_len` | `int` | `10000` | Max ORF length in nt (caps payloads); raise to 1_000_000_000 for genome-scale |
| `include_stop` | `bool` | `True` | Include the stop codon in the reported ORF nucleotide sequence and length |
| `ignore_case` | `bool` | `False` | Treat lowercase (soft-masked) nucleotides as ORF-eligible |
| `partial_3` | `bool` | `False` | Report ORFs missing a stop codon at the sequence end |
| `partial_5` | `bool` | `False` | Report ORFs missing a start codon at the sequence start |
| `between_stops` | `bool` | `False` | Report ORFs spanning stop-to-stop (ignores start codons; implies partial_3 + partial_5) |
| `translation_table` | `Literal['standard', 'vertebrate_mitochondrial', 'yeast_mitochondrial', 'mold_protozoan_mitochondrial', 'invertebrate_mitochondrial', 'ciliate_nuclear', 'echinoderm_mitochondrial', 'euplotid_nuclear', 'bacterial', 'alternative_yeast_nuclear', 'ascidian_mitochondrial', 'alternative_flatworm_mitochondrial', 'chlorophycean_mitochondrial', 'trematode_mitochondrial', 'scenedesmus_mitochondrial', 'thraustochytrium_mitochondrial', 'rhabdopleuridae_mitochondrial', 'candidate_division_sr1', 'pachysolen_nuclear', 'karyorelict_nuclear', 'condylostoma_nuclear', 'mesodinium_nuclear', 'peritrich_nuclear'] \| None` | `None` | NCBI genetic code for translation (None = standard genetic code) |

In [6]:
# Run the prediction
result = run_orfipy_prediction(inputs, config)

Running run_orfipy_prediction [00:00]

In [7]:
# Display output docs
display_api_reference("orfipy", "output", "run_orfipy_prediction")

# Inspect predicted ORFs from the single input sequence
print(f"Found {result.num_orfs} ORFs")
for orf_list in result.predicted_orfs:
    for orf in orf_list:
        print(f"  {orf.id}: {orf.amino_acid_sequence} ({orf.amino_acid_length} aa, {orf.strand} strand, frame {orf.frame})")

# Demonstrate batch processing with custom sequence IDs
sequences = [
    "ATGCCCAAATTTGGGCCCAAATTTGGGCCCAAATTTGGGTAG",
    "ATGTTTCCCGGGAAATTTCCCGGGTAA",
    "ATGAAACCCGGGAAATTTCCCGGGAAATTTCCCGGGAAATTTCCCGGGTAG",
]
batch_inputs = OrfipyInput(
    sequences=sequences,
)
batch_config = OrfipyConfig(min_len=12)
batch_result = run_orfipy_prediction(batch_inputs, batch_config)

print(f"\nBatch: {batch_result.num_orfs} total ORFs across {len(sequences)} sequences")
print(f"ORFs per sequence: {batch_result.num_orfs_per_sequence}")

# Inspect an individual ORF
orf = batch_result.predicted_orfs[0][0]
print(f"\nFirst ORF details:")
print(f"  Parent ID:   {orf.parent_id}")
print(f"  Position:    {orf.nucleotide_start}-{orf.nucleotide_end}")
print(f"  AA length:   {orf.amino_acid_length}")
print(f"  GC content:  {orf.gc_content:.1f}%")
print(f"  Protein:     {orf.amino_acid_sequence}")

**Output** — `OrfipyOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `predicted_orfs` | `list[list[proto_tools.tools.orf_prediction.orf.ORF]]` | `[]` | List of ORF results per input sequence |

Found 1 ORFs
  seq_0_ORF.1: MVLSPADKTNVKAAWGK (17 aa, + strand, frame 1)


Running run_orfipy_prediction [00:00]


Batch: 3 total ORFs across 3 sequences
ORFs per sequence: [1, 1, 1]

First ORF details:
  Parent ID:   gene_alpha
  Position:    1-42
  AA length:   13
  GC content:  47.6%
  Protein:     MPKFGPKFGPKFG


## Export Results

Predicted ORFs can be exported to CSV or JSON for tabular analysis, or to FASTA format (FAA for protein sequences, FNA for nucleotide sequences) for use in downstream bioinformatics pipelines.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

batch_result.export("orfipy_results", export_path=output_dir, file_format="csv")
print(f"Exported CSV to {output_dir / 'orfipy_results.csv'}")

batch_result.export("orfipy_results", export_path=output_dir, file_format="faa")
print(f"Exported protein FASTA to {output_dir / 'orfipy_results.faa'}")

Exported CSV to example_output/orfipy_results.csv
Exported protein FASTA to example_output/orfipy_results.faa
